In [5]:
import msl_reader

In [8]:
fn = "SpectralLibrary_2026-03-23-16-18-01.msl"

In [11]:
msl_reader.load_index_only(fn)

ValueError: Trailing magic mismatch: expected 0x4D5A4C42, got 0x424C5A4D. File may be truncated or corrupt.

In [12]:
from msl_reader import *

In [27]:
with open(fn, 'rb') as f:
        # ── 1. Read and validate header magic ────────────────────────────────
        magic_bytes = f.read(4)
        if magic_bytes != HEADER_MAGIC:
            raise ValueError(
                f'Invalid .msl magic: expected {list(HEADER_MAGIC)}, '
                f'got {list(magic_bytes)}'
            )

        # ── 2. Read full header ───────────────────────────────────────────────
        f.seek(0)
        header_size = struct.calcsize(HEADER_FMT)  # 64
        header_raw = f.read(header_size)
        if len(header_raw) < header_size:
            raise ValueError('File too short to contain a valid .msl header')

        (magic, format_version, file_flags, n_precursors, n_proteins,
         n_elution_groups, n_strings, reserved,
         protein_table_offset, string_table_offset,
         precursor_section_offset, fragment_section_offset) = \
            struct.unpack(HEADER_FMT, header_raw)

        # ── 2. Validate format version ────────────────────────────────────────
        if not (MIN_SUPPORTED_VERSION <= format_version <= MAX_SUPPORTED_VERSION):
            raise ValueError(
                f'Unsupported .msl format version: {format_version} '
                f'(supported: {MIN_SUPPORTED_VERSION}–{MAX_SUPPORTED_VERSION})'
            )

        # ── 3. Read footer (last 20 bytes) and validate trailing magic ────────
        footer_size = struct.calcsize(FOOTER_FMT)  # 20
        f.seek(-footer_size, 2)
        footer_raw = f.read(footer_size)
        if len(footer_raw) < footer_size:
            raise ValueError('File too short to contain a valid .msl footer')

        (offset_table_offset, footer_n_precursors,
         data_crc32, trailing_magic) = struct.unpack(FOOTER_FMT, footer_raw)

In [28]:
trailing_magic , FOOTER_MAGIC_UINT32

(1112300109, 1297763394)

TypeError: unpack expected 2 arguments, got 1